# 09 — FAIR1M fine-grained finetune

End-to-end tutorial for **fine-grained oriented detection** on the public FAIR1M train dump: 37 classes such as Boeing 737 vs A321, or dump truck vs cargo truck, instead of DOTA's 15. It is also the full data-preparation lesson: convert a raw dataset to DOTA format, hold out validation images, tile, and fine-tune from a DOTA Hub checkpoint.

| | |
|---|---|
| Dataset | [`ollypowell/fair1m-satellite-imagery-for-object-detection`](https://www.kaggle.com/datasets/ollypowell/fair1m-satellite-imagery-for-object-detection) (~9 GB JPG) |
| License | **CC BY-NC-SA 3.0 IGO** (non-commercial). OrientedDet does **not** ship Hub weights for FAIR1M. |
| Local | `FAIR1M_ROOT` → the raw dump (`Dataset/Images/Train`, ...) |
| GPU | Only for the training smoke (`RUN_TRAIN = True`); CPU is enough for convert and tile |
| Post | [FAIR1M fine-grained oriented detection](https://deeplearning.earth/posts/2026-09-24_fair1m_fine_grained_oriented_detection/) |

### Kaggle setup

1. **Add Input** → `fair1m-satellite-imagery-for-object-detection` (ollypowell)
2. **Accelerator** → GPU (T4/P100) if you enable the training smoke
3. **Internet** → ON (for `pip install` and the DOTA checkpoint)
4. Defaults use a **small image subset** so convert + tile fit in `/kaggle/working`. Set `SUBSET_IMAGES = None` only if you have enough disk and time.

**Pipeline:** discover → visualize → `export_fair1m_to_dota` (image-level holdout) → `odet tile-dota` (1024 / overlap 200) → short `odet train` smoke from `hf://rotated_faster_rcnn_dota_le90_1x`.

**Do not expect DOTA-like mAP.** A full 1× Rotated Faster R-CNN finetune from the DOTA Hub checkpoint reached **36.7%** mAP50 on tiled val (still climbing). Literature Faster R-CNN sits in the low-to-mid 30s. The smoke run will not approach that number — see **Expected mAP** below.

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd), flush=True)
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. Paths & knobs

In [ ]:
import json
import shutil
from collections import Counter

from PIL import Image

RUN_TRAIN = False        # explore-only by default; True runs a short training smoke
SUBSET_IMAGES = 48       # None = full train dump (slow, disk-heavy)
VAL_FRACTION = 0.1
SPLIT_SEED = 0
TILE_SIZE = 1024
TILE_OVERLAP = 200       # pixels
MIN_BOX_OVERLAP = 0.7
MAX_TRAIN_SAMPLES = 64   # tile caps for the smoke run
MAX_VAL_SAMPLES = 16
NUM_EPOCHS = 1

DATA_ROOT = kaggle_input("fair1m") or Path(os.environ.get("FAIR1M_ROOT", "/path/to/data/FAIR1M"))
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"FAIR1M not found at {DATA_ROOT}. On Kaggle add "
                            "ollypowell/fair1m-satellite-imagery-for-object-detection; locally set FAIR1M_ROOT.")
F_WORK = WORK / "09_fair1m"
F_WORK.mkdir(parents=True, exist_ok=True)
DOTA_DIR = F_WORK / "FAIR1M-dota"
print("DATA_ROOT =", DATA_ROOT)
print("F_WORK    =", F_WORK, f"(free ≈ {shutil.disk_usage(F_WORK).free / 1e9:.0f} GB)")

## 2. Discover layout & class counts

In [ ]:
from oriented_det.data import FAIR1M_CLASSES, FAIR1M_GROUPS
from oriented_det.data.fair1m import FAIR1MDataset, discover_fair1m_pairs, resolve_fair1m_root

print(f"{len(FAIR1M_CLASSES)} fine classes in {len(FAIR1M_GROUPS)} coarse groups")
root = resolve_fair1m_root(DATA_ROOT)
pairs = [(img, xml) for img, xml in discover_fair1m_pairs(DATA_ROOT, "train") if xml is not None]
print(f"Resolved root: {root}   labelled train images: {len(pairs)}")
if not pairs:
    raise RuntimeError(
        "No image/XML pairs found. Expect Dataset/Images/Train/*.jpg with either "
        "Dataset/Labels/Train/*.xml (same stem) or Notebook_Working/train_labels/N.xml "
        "(JPG stem t_N → XML stem N)."
    )

scan_n = min(len(pairs), SUBSET_IMAGES or 200, 200)
ds_scan = FAIR1MDataset(DATA_ROOT, split="train", difficult_strategy="keep",
                        stem_list=[p[0].stem for p in pairs[:scan_n]])
counts = Counter(ann.class_name for sample in ds_scan for ann in sample.annotations)
print(f"Scanned {scan_n} images → {sum(counts.values())} boxes, {len(counts)} classes")
for name, n in counts.most_common(15):
    print(f"  {n:6d}  {name}")
if len(counts) > 15:
    print(f"  ... ({len(counts) - 15} more)")

## 3. Visualize oriented boxes

In [ ]:
from oriented_det.geometry import RBox
from oriented_det.utils.viz import DrawingSpec, draw_boxes

views, titles = [], []
for sample in FAIR1MDataset(DATA_ROOT, split="train", difficult_strategy="keep", stem_list=[p[0].stem for p in pairs[:3]]):
    img = Image.open(sample.image_path).convert("RGB")
    scale = min(1.0, 1024 / max(img.size))
    img = img.resize((int(img.width * scale), int(img.height * scale)), Image.BILINEAR)
    boxes = [RBox(a.rbox.cx * scale, a.rbox.cy * scale, a.rbox.width * scale, a.rbox.height * scale, a.rbox.angle)
             for a in sample.annotations[:80]]
    views.append(draw_boxes(img, boxes, specs=[DrawingSpec(outline=(255, 200, 0), width=2)] * len(boxes)))
    titles.append(f"{sample.image_path.name}: {len(sample.annotations)} objects")
show(*views, titles=titles, size=5)

## 4. Optional subset (symlinks)

Builds a tiny Kaggle-shaped tree under working storage so convert and tile stay small.

In [ ]:
def build_symlink_subset(src_root, dst_root, n):
    pairs_all = sorted(discover_fair1m_pairs(src_root, "train"), key=lambda t: t[0].stem)
    pairs_all = [(img, xml) for img, xml in pairs_all if xml is not None][: int(n)]
    if len(pairs_all) < 2:
        raise ValueError("Need at least 2 labelled images for a train/val holdout")
    if dst_root.exists():
        shutil.rmtree(dst_root)
    img_out = dst_root / "Dataset" / "Images" / "Train"
    lbl_out = dst_root / "Dataset" / "Labels" / "Train"
    img_out.mkdir(parents=True)
    lbl_out.mkdir(parents=True)
    for img_path, xml_path in pairs_all:
        os.symlink(img_path.resolve(), img_out / img_path.name)
        # Same-stem label name so the subset tree works without Notebook_Working.
        os.symlink(xml_path.resolve(), lbl_out / f"{img_path.stem}.xml")
    print(f"Subset: {len(pairs_all)} pairs → {dst_root}")
    return dst_root


EXPORT_ROOT = DATA_ROOT if SUBSET_IMAGES is None else build_symlink_subset(DATA_ROOT, F_WORK / "fair1m_subset", SUBSET_IMAGES)

## 5. Convert → DOTA PNG + image-level holdout

Split **images** before tiling, so overlapping tiles of one image cannot leak across train and val. This is the mistake behind DOTA's leaky eval-val (lesson 07), avoided by construction.

In [ ]:
from oriented_det.data.fair1m import export_fair1m_to_dota

if DOTA_DIR.exists():
    shutil.rmtree(DOTA_DIR)
exported = export_fair1m_to_dota(EXPORT_ROOT, DOTA_DIR, splits=("train", "val"), difficult_strategy="keep",
                                 val_fraction=VAL_FRACTION, split_seed=SPLIT_SEED)
print("Exported:", exported)
print("ImageSets:", sorted(p.name for p in (DOTA_DIR / "ImageSets").glob("*.txt")))

## 6. Tile (1024 / overlap 200)

The same tiling rule as sliding-window inference (lesson 06), applied to labelled data. `--min-overlap` keeps a box in a tile only if at least that fraction of it lies inside.

In [ ]:
for split in ("train", "val"):
    run([*ODET, "tile-dota", DOTA_DIR / split, "--tile-size", TILE_SIZE, "--overlap", TILE_OVERLAP,
         "--min-overlap", MIN_BOX_OVERLAP, "--overwrite"])

TRAIN_TILES = DOTA_DIR / "train" / f"tiles_{TILE_SIZE}"
VAL_TILES = DOTA_DIR / "val" / f"tiles_{TILE_SIZE}"
for label, d in (("train", TRAIN_TILES), ("val", VAL_TILES)):
    n_img = len(list((d / "images").iterdir())) if (d / "images").is_dir() else 0
    n_txt = len(list((d / "labels").glob("*.txt"))) if (d / "labels").is_dir() else 0
    print(f"{label}: {n_img} images, {n_txt} labels → {d}")

## 7. Smoke train (capped samples)

Patch the published recipe (`rotated_faster_rcnn/fair1m_le90_1x.json`) the way lesson 07 showed: point it at the tiles, cap the samples, one epoch. The recipe initialises from `hf://rotated_faster_rcnn_dota_le90_1x` and re-initialises the classifier for 37 classes. This checks the convert → tile → train path; it is **not** the 12-epoch 36.7% run.

In [ ]:
from oriented_det.train.config import TrainingExperimentConfig

cfg = TrainingExperimentConfig.load(CONFIGS / "rotated_faster_rcnn/fair1m_le90_1x.json")
cfg.dataset.data_root = DOTA_DIR
cfg.dataset.train_tiles_dir = TRAIN_TILES
cfg.dataset.val_tiles_dir = VAL_TILES
cfg.dataset.max_train_samples = MAX_TRAIN_SAMPLES
cfg.dataset.max_val_samples = MAX_VAL_SAMPLES
cfg.training.num_epochs = NUM_EPOCHS
cfg.evaluation.compute_map_final = False
cfg.evaluation.compute_map_every_n_epochs = 0
CONFIG_PATH = F_WORK / "fair1m_smoke.json"
cfg.save(CONFIG_PATH)
print("Wrote", CONFIG_PATH, "| init from", cfg.checkpoint.load_from_checkpoint)

In [ ]:
if not RUN_TRAIN:
    print("RUN_TRAIN=False: skipping training")
else:
    run([*ODET, "train", "--config", CONFIG_PATH])
    print("Checkpoints:")
    for p in sorted((WORK / "runs").rglob("*.pth"))[-5:]:
        print(" ", p)

## Expected mAP (full 1× Faster R-CNN)

A complete 12-epoch finetune of [`configs/rotated_faster_rcnn/fair1m_le90_1x.json`](https://github.com/DL4EO/oriented-det/blob/main/configs/rotated_faster_rcnn/fair1m_le90_1x.json) from `hf://rotated_faster_rcnn_dota_le90_1x` (37-way classifier re-init) finished at **36.70%** mAP50 on non-empty val tiles.

Run `runs/rotated_faster_rcnn/20260910-072116` (NVIDIA L4, ~24.5 h). Train 20,055 tiles / val 9,896 after `filter_empty_gt`. Periodic mAP every 4 epochs, score ≥ 0.05, rotated IoU 0.50 (exact CPU polygon). **Not** the Gaofen hidden test and **not** FAIR1M `mAP_F`. No FAIR1M Hub zoo.

| Epoch | Train loss | Val mAP50 |
|------:|-----------:|----------:|
| 4 | 0.496 | 30.03% |
| 8 | 0.456 | 33.71% |
| 12 | 0.412 | **36.70%** |

That looks low next to DOTA (same detector ~74% official Task 1 / ~83% leaky eval-val) and is **in band for FAIR1M**:

- FAIR1M paper, Faster R-CNN R101, official OBB: **31.53%**
- Later Rotated Faster R-CNN R50 papers: **~33–35%**
- Oriented R-CNN R50 (rotated proposals): **~39–42%**

The bottleneck is **class ID, not boxes**. Epoch 12 mean best IoU vs any detection was **0.62**, same-class **0.50**, GT cover **62%** (DOTA ~88%). Train imbalance is **1038×** (Small Car 143k vs C919 138). Rare subtypes stay near 0 AP (Tractor, other-ship, Trailer, ARJ21, C919); sports fields are easy (Baseball Field 88.5%, Tennis Court 81%). Loss and mAP were still moving at epoch 12.

To raise it: resume / 3×; switch to Oriented R-CNN 1×; enable `loss.roi_grouped_ce_*` or class weights for airplane / ship / vehicle subtypes. Details: [FAIR1M user guide](https://github.com/DL4EO/oriented-det/blob/main/docs/user-guide/data.md#fair1m).

## Notes and what to try next

- **Holdout protocol:** sorted stems, `md5(f"{seed}:{stem}")` vs `val_fraction` (defaults seed `0`, fraction `0.1`). Lists land in `FAIR1M-dota/ImageSets/`. Official FAIR1M-1.0 already has labelled val — omit `val_fraction` on a full dump.
- **Zoo roles:** DOTA = Hub pretrain; HRSC2016 = published small-data Hub; FAIR1M = train/metrics support only (this notebook).
- **Full run:** set `SUBSET_IMAGES = None`, drop the sample caps, keep 12 epochs, and prefer a machine with a large disk — Kaggle working space is usually too small for the full tile set. Expect mid-30s mAP50 for Faster R-CNN 1×, not DOTA 70%+.
- Evaluate your run with `odet preds --experiment-dir runs/rotated_faster_rcnn/<timestamp> --data-split val` (lesson 07).
- **Next:** [10 — SAR ships: SSDD and HRSID](10_sar_ssdd_hrsid.ipynb).